# Stores Bronze to Silver Transformation

## Objective:
### Transform Stores Information data from the Bronze layer into the Silver layer by applying data quality validations checks, business rule validations,and before storing the cleansed data in ADLS Gen2.
### Source: Bronze (CSV)
### Destination: Silver (Parquet)
### Technologies: Azure Databricks, PySpark, ADLS Gen2, Parquet

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
display(
    spark.read
         .option("header", "true")
         .csv("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/stores/stores_final.csv")
)

Store_ID,Store_Name,Country,State,City,Store_Type,Employee_Count,Opening_Date
S001,Pune Retail 1,India,Maharashtra,Pune,Outlet,81,2025-03-06
S002,Nashik Retail 2,India,Maharashtra,Nashik,Standalone,84,2024-08-22
S003,Nashik Retail 3,India,Maharashtra,Nashik,Standalone,59,2020-06-02
S004,Pune Retail 4,India,Maharashtra,Pune,Outlet,86,2018-01-19
S005,Mumbai Retail 5,India,Maharashtra,Mumbai,Standalone,98,2018-03-07
S006,Mumbai Retail 6,India,Maharashtra,Mumbai,Standalone,48,2024-06-06
S007,Nashik Retail 7,India,Maharashtra,Nashik,Flagship,40,2021-04-04
S008,Nashik Retail 8,India,Maharashtra,Nashik,Flagship,71,2023-02-08
S009,Pune Retail 9,India,Maharashtra,Pune,Outlet,48,2023-09-22
S010,Pune Retail 10,India,Maharashtra,Pune,Outlet,35,2024-01-01


In [0]:
store_schema = '''
Store_ID STRING,
Store_Name STRING,
Country STRING,
State STRING,
City STRING,
Store_Type STRING,
Employee_Count INTEGER,
Opening_Date DATE
'''

In [0]:
print(store_schema)


Store_ID STRING,
Store_Name STRING,
Country STRING,
State STRING,
City STRING,
Store_Type STRING,
Employee_Count INTEGER,
Opening_Date DATE



In [0]:
df_stores = spark.read.schema(store_schema)\
            .format("csv").option("header", "true")\
            .load("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/stores/stores_final.csv")

display(df_stores)
df_stores.printSchema()

Store_ID,Store_Name,Country,State,City,Store_Type,Employee_Count,Opening_Date
S001,Pune Retail 1,India,Maharashtra,Pune,Outlet,81,2025-03-06
S002,Nashik Retail 2,India,Maharashtra,Nashik,Standalone,84,2024-08-22
S003,Nashik Retail 3,India,Maharashtra,Nashik,Standalone,59,2020-06-02
S004,Pune Retail 4,India,Maharashtra,Pune,Outlet,86,2018-01-19
S005,Mumbai Retail 5,India,Maharashtra,Mumbai,Standalone,98,2018-03-07
S006,Mumbai Retail 6,India,Maharashtra,Mumbai,Standalone,48,2024-06-06
S007,Nashik Retail 7,India,Maharashtra,Nashik,Flagship,40,2021-04-04
S008,Nashik Retail 8,India,Maharashtra,Nashik,Flagship,71,2023-02-08
S009,Pune Retail 9,India,Maharashtra,Pune,Outlet,48,2023-09-22
S010,Pune Retail 10,India,Maharashtra,Pune,Outlet,35,2024-01-01


root
 |-- Store_ID: string (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Employee_Count: integer (nullable = true)
 |-- Opening_Date: date (nullable = true)



## Data Profiling

### Null Value analysis

In [0]:
df_total_null_count= df_stores.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_stores.columns
    ]
)
df_total_null_count.show()

+--------+----------+-------+-----+----+----------+--------------+------------+
|Store_ID|Store_Name|Country|State|City|Store_Type|Employee_Count|Opening_Date|
+--------+----------+-------+-----+----+----------+--------------+------------+
|       0|         0|      0|    0|   0|         0|             0|           0|
+--------+----------+-------+-----+----+----------+--------------+------------+



### Duplicate Value analysis

In [0]:
df_dup_store_count = df_stores.groupBy("Store_ID").count()
df_dup_store_count = df_dup_store_count.filter(col("count") > 1)
df_dup_store_count.display()

Store_ID,count


### Business rule validation

In [0]:
df_invalid_store_id = df_stores.filter(col("Store_ID").isNull())
df_invalid_store_id.show()
df_invalid_store_name = df_stores.filter(col("Store_Name").isNull())
df_invalid_store_name.show()

+--------+----------+-------+-----+----+----------+--------------+------------+
|Store_ID|Store_Name|Country|State|City|Store_Type|Employee_Count|Opening_Date|
+--------+----------+-------+-----+----+----------+--------------+------------+
+--------+----------+-------+-----+----+----------+--------------+------------+

+--------+----------+-------+-----+----+----------+--------------+------------+
|Store_ID|Store_Name|Country|State|City|Store_Type|Employee_Count|Opening_Date|
+--------+----------+-------+-----+----+----------+--------------+------------+
+--------+----------+-------+-----+----+----------+--------------+------------+



### verifying if all columns are not null

In [0]:
df_invalid_store_details = df_stores.filter(

    col("Store_ID").isNull() |
    col("Store_Name").isNull() |
    col("Country").isNull() |
    col("State").isNull() |
    col("City").isNull() |
    col("Store_Type").isNull()
)

df_invalid_store_details.show()

+--------+----------+-------+-----+----+----------+--------------+------------+
|Store_ID|Store_Name|Country|State|City|Store_Type|Employee_Count|Opening_Date|
+--------+----------+-------+-----+----+----------+--------------+------------+
+--------+----------+-------+-----+----+----------+--------------+------------+



In [0]:
df_valid_emp_count = df_stores.filter(col("Employee_Count").isNull())
df_valid_emp_count.show()


    

+--------+----------+-------+-----+----+----------+--------------+------------+
|Store_ID|Store_Name|Country|State|City|Store_Type|Employee_Count|Opening_Date|
+--------+----------+-------+-----+----+----------+--------------+------------+
+--------+----------+-------+-----+----+----------+--------------+------------+



In [0]:
df_valid_launch_date = df_stores.filter(col("Opening_Date") < current_date())
df_valid_launch_date.show()

+--------+-------------------+-------+--------------+---------+----------+--------------+------------+
|Store_ID|         Store_Name|Country|         State|     City|Store_Type|Employee_Count|Opening_Date|
+--------+-------------------+-------+--------------+---------+----------+--------------+------------+
|    S001|      Pune Retail 1|  India|   Maharashtra|     Pune|    Outlet|            81|  2025-03-06|
|    S002|    Nashik Retail 2|  India|   Maharashtra|   Nashik|Standalone|            84|  2024-08-22|
|    S003|    Nashik Retail 3|  India|   Maharashtra|   Nashik|Standalone|            59|  2020-06-02|
|    S004|      Pune Retail 4|  India|   Maharashtra|     Pune|    Outlet|            86|  2018-01-19|
|    S005|    Mumbai Retail 5|  India|   Maharashtra|   Mumbai|Standalone|            98|  2018-03-07|
|    S006|    Mumbai Retail 6|  India|   Maharashtra|   Mumbai|Standalone|            48|  2024-06-06|
|    S007|    Nashik Retail 7|  India|   Maharashtra|   Nashik|  Flagship

In [0]:
df_trim_stores = df_stores\
                    .withColumn("Store_ID", trim(col("Store_ID")))\
                    .withColumn("Store_Name", trim(col("Store_Name")))\
                    .withColumn("Country", trim(col("Country")))\
                    .withColumn("State", trim(col("State")))\
                    .withColumn("City", trim(col("City")))\
                    .withColumn("Store_Type", trim(col("Store_Type")))\
                   
                           

                        

display(df_trim_stores.limit(10))

Store_ID,Store_Name,Country,State,City,Store_Type,Employee_Count,Opening_Date
S001,Pune Retail 1,India,Maharashtra,Pune,Outlet,81,2025-03-06
S002,Nashik Retail 2,India,Maharashtra,Nashik,Standalone,84,2024-08-22
S003,Nashik Retail 3,India,Maharashtra,Nashik,Standalone,59,2020-06-02
S004,Pune Retail 4,India,Maharashtra,Pune,Outlet,86,2018-01-19
S005,Mumbai Retail 5,India,Maharashtra,Mumbai,Standalone,98,2018-03-07
S006,Mumbai Retail 6,India,Maharashtra,Mumbai,Standalone,48,2024-06-06
S007,Nashik Retail 7,India,Maharashtra,Nashik,Flagship,40,2021-04-04
S008,Nashik Retail 8,India,Maharashtra,Nashik,Flagship,71,2023-02-08
S009,Pune Retail 9,India,Maharashtra,Pune,Outlet,48,2023-09-22
S010,Pune Retail 10,India,Maharashtra,Pune,Outlet,35,2024-01-01


In [0]:
df_silver_stores = (
    df_trim_stores.write
        .format("parquet")
        .mode("overwrite")
        .save("abfss://silver@stretaillakehouse01.dfs.core.windows.net/stores/")
)

In [0]:
df_silver_stores = (
    spark.read
         .format("parquet")
         .load("abfss://silver@stretaillakehouse01.dfs.core.windows.net/stores/")
)

In [0]:
df_silver_stores.display()
df_silver_stores.printSchema()

Store_ID,Store_Name,Country,State,City,Store_Type,Employee_Count,Opening_Date
S001,Pune Retail 1,India,Maharashtra,Pune,Outlet,81,2025-03-06
S002,Nashik Retail 2,India,Maharashtra,Nashik,Standalone,84,2024-08-22
S003,Nashik Retail 3,India,Maharashtra,Nashik,Standalone,59,2020-06-02
S004,Pune Retail 4,India,Maharashtra,Pune,Outlet,86,2018-01-19
S005,Mumbai Retail 5,India,Maharashtra,Mumbai,Standalone,98,2018-03-07
S006,Mumbai Retail 6,India,Maharashtra,Mumbai,Standalone,48,2024-06-06
S007,Nashik Retail 7,India,Maharashtra,Nashik,Flagship,40,2021-04-04
S008,Nashik Retail 8,India,Maharashtra,Nashik,Flagship,71,2023-02-08
S009,Pune Retail 9,India,Maharashtra,Pune,Outlet,48,2023-09-22
S010,Pune Retail 10,India,Maharashtra,Pune,Outlet,35,2024-01-01


root
 |-- Store_ID: string (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Employee_Count: integer (nullable = true)
 |-- Opening_Date: date (nullable = true)



In [0]:
df_silver_stores.count()

25

In [0]:
df_stores.count()

25